# Credit Card Fraud Detection — Mini Project
**Domain:** Financial Fraud Analytics

**Problem Statement:** Financial institutions face significant losses from fraudulent transactions. This project transforms raw, anonymized transaction records into structured business intelligence to evaluate fraud patterns, transaction volume, and risk exposure across Power BI and Tableau.

**Dataset:** Credit Card Fraud Detection (Kaggle — mlg-ulb/creditcardfraud)
- 284,807 transactions, 492 labeled frauds (0.172%)
- `Time`, `Amount` are raw; `V1`–`V28` are PCA-anonymized features; `Class` is the fraud label (1 = fraud, 0 = legit)

In [4]:
!pip install pandas sqlalchemy pymysql


In [ ]:
import pandas as pd

file_path = "creditcard.csv"

df = pd.read_csv(file_path)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()


Rows: 284807
Columns: 31


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [6]:
# Make a copy for processing
cc = df.copy()

# --------------------------------------------------
# 1. Check for missing values (this dataset is usually clean,
#    but we verify rather than assume)
# --------------------------------------------------
print("Missing values per column:\n", cc.isnull().sum().sum())

# --------------------------------------------------
# 2. Remove duplicate records
# --------------------------------------------------
before = cc.shape[0]
cc.drop_duplicates(inplace=True)
print(f"Removed {before - cc.shape[0]} duplicate rows")


Missing values per column:
 0
Removed 1081 duplicate rows


In [7]:
# --------------------------------------------------
# Feature engineering — derive a time dimension from "Time"
# (Time = seconds elapsed since the first transaction in the dataset)
# --------------------------------------------------
cc["transaction_hour"] = ((cc["Time"] // 3600) % 24).astype(int)
cc["transaction_day"] = (cc["Time"] // (3600 * 24)).astype(int) + 1  # Day 1 or Day 2

# Categorize transaction size for easier BI filtering
cc["amount_category"] = pd.cut(
    cc["Amount"],
    bins=[-0.01, 10, 50, 200, 1000, cc["Amount"].max()],
    labels=["<10", "10-50", "50-200", "200-1000", "1000+"]
)

# Human-readable fraud label
cc["fraud_label"] = cc["Class"].map({0: "Legitimate", 1: "Fraud"})

# Standardize column names
cc.columns = (
    cc.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

# --------------------------------------------------
# Filter out invalid rows (negative amounts shouldn't exist, but verify)
# --------------------------------------------------
cc = cc[cc["amount"] >= 0]

print("Original Shape:", df.shape)
print("Cleaned Shape:", cc.shape)
cc.head()


Original Shape: (284807, 31)
Cleaned Shape: (283726, 35)


,time,v1,v2,v3,v4,v5,v6,v7,v8,v9,...,v25,v26,v27,v28,amount,class,transaction_hour,transaction_day,amount_category,fraud_label
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.128539,-0.189115,0.133558,-0.021053,149.62,0,0,1,50-200,Legitimate
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0.167170,0.125895,-0.008983,0.014724,2.69,0,0,1,<10,Legitimate
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0,0,1,200-1000,Legitimate
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0.647376,-0.221929,0.062723,0.061458,123.50,0,0,1,50-200,Legitimate
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.206010,0.502292,0.219422,0.215153,69.99,0,0,1,50-200,Legitimate


In [8]:
# Quick class balance check — this is the key insight of this dataset
print(cc["fraud_label"].value_counts())
print(cc["fraud_label"].value_counts(normalize=True) * 100)


fraud_label
Legitimate    283253
Fraud            473
Name: count, dtype: int64
fraud_label
Legitimate    99.83329
Fraud          0.16671
Name: proportion, dtype: float64


## MySQL — Database and Data Loading

In [18]:
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

username = "root"
password = "Shreya@5s27"
safe_password = quote_plus(password)
host = "127.0.0.1"
port = 3306

In [19]:
server_engine = create_engine(
    f"mysql+pymysql://{username}:{safe_password}@{host}:{port}"
)

with server_engine.connect() as connection:
    connection.execute(
        text("CREATE DATABASE IF NOT EXISTS fraud_analytics")
    )

print("Database created/exists.")


Database created/exists.


In [20]:
print(repr(username), repr(password), repr(host), repr(port))

'root' 'Shreya@5s27' '127.0.0.1' 3306


In [21]:
db_engine = create_engine(
    f"mysql+pymysql://{username}:{safe_password}@{host}:{port}/fraud_analytics"
)

print("Connected to fraud_analytics!")


Connected to fraud_analytics!


In [27]:
cc.to_sql(
    "transactions",
    con=db_engine,
    if_exists="replace",
    index=False,
    chunksize=5000
)

print("Data loaded successfully!")


Data loaded successfully!


In [23]:
pd.read_sql(
    "SELECT COUNT(*) AS total_rows FROM transactions",
    db_engine
)


,total_rows
0,283726


In [26]:
pd.read_sql(
    "SELECT * FROM transactions LIMIT 5",
    db_engine
)


,time,v1,v2,v3,v4,v5,v6,v7,v8,v9,...,v25,v26,v27,v28,amount,class,transaction_hour,transaction_day,amount_category,fraud_label
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.128539,-0.189115,0.133558,-0.021053,149.62,0,0,1,50-200,Legitimate
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0.167170,0.125895,-0.008983,0.014724,2.69,0,0,1,<10,Legitimate
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0,0,1,200-1000,Legitimate
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0.647376,-0.221929,0.062723,0.061458,123.50,0,0,1,50-200,Legitimate
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.206010,0.502292,0.219422,0.215153,69.99,0,0,1,50-200,Legitimate


In [28]:
# Quick sanity-check — these numbers should match the KPI cards
# you'll later build in Power BI / Tableau
summary = pd.read_sql("""
    SELECT
        COUNT(*) AS total_transactions,
        SUM(class) AS total_fraud_cases,
        ROUND(100 * SUM(class) / COUNT(*), 4) AS fraud_rate_pct,
        ROUND(AVG(amount), 2) AS avg_transaction_amount,
        ROUND(SUM(amount), 2) AS total_transaction_volume
    FROM transactions
""", db_engine)

summary


,total_transactions,total_fraud_cases,fraud_rate_pct,avg_transaction_amount,total_transaction_volume
0,283726,473.0,0.1667,88.47,25102001.68
